# 🎨 **ColabRP: Image Backend (Stable Diffusion XL / Illustrious)**

Este notebook instala y ejecuta **Stable Diffusion WebUI (A1111)** optimizado para Google Colab T4.

### **Novedades (Febrero 2026):**
- ✅ Soporte completo para **Python 3.12**.
- ✅ Modelo **Illustrious XL v1.0** preconfigurado.
- ✅ Nueva optimización **SDP Attention** (más estable que xformers en Colab).
- ✅ Instalación manual de repositorios para evitar errores de Git.

In [ ]:
# @title 1. Configuración
# @markdown ### 🤖 Modelo Base
MODEL_TYPE = "Anime (Illustrious XL)" # @param ["Anime (Illustrious XL)", "Realismo (RealisticVision)", "General (SD 1.5 Base)"]

# @markdown ### 🧩 LoRA Predefinido
LORA_SELECT = "Todas (Descargar las 3)" # @param ["Todas (Descargar las 3)", "Nemu Manaka", "Sharona (Blue Archive)", "Blonde Sailor (OC)", "Ninguna"]

# @markdown ### ☁️ Sincronización (npoint.io)
NPOINT_ID = "9b9221cf0ddd4db77078" # @param {type:"string"}

import os, time, requests, subprocess, re, threading, json

def log(msg):
    print(f"[ImageBackend] {msg}")

model_map = {
    "Anime (Illustrious XL)": "https://huggingface.co/Liberata/illustrious-xl-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors",
    "Realismo (RealisticVision)": "https://huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE/resolve/main/Realistic_Vision_V5.1_fp16-no-ema.safetensors",
    "General (SD 1.5 Base)": "https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"
}

lora_map = {
    "Nemu Manaka": {
        "url": "https://civitai.com/api/download/models/1245821?type=Model&format=SafeTensor",
        "triggers": "<lora:NemuManaka-IL-v1-08:0.8>, ChopioNemu, brown hair, short hair, french braid, single hair bun, blunt bangs, purple eyes, black serafuku"
    },
    "Sharona (Blue Archive)": {
        "url": "https://civitai.com/api/download/models/1665610?type=Model&format=SafeTensor",
        "triggers": "sharona, 1girl, hairband, breasts, solo, lips, school uniform, sweater vest, skirt"
    },
    "Blonde Sailor (OC)": {
        "url": "https://civitai.com/api/download/models/1735183?type=Model&format=SafeTensor",
        "triggers": "blonde hair, long hair, brown eyes, Green hairband, large breasts, black choker, Sailor collar, Yellow cardigan"
    }
}

In [ ]:
# @title 2. Instalar y Descargar
log("🚀 Iniciando instalación completa...")

# 1. Herramientas Base
!apt-get update && apt-get install -y aria2 git libgl1-mesa-glx

# 2. Clonar WebUI
if not os.path.exists("stable-diffusion-webui"):
    log("📦 Clonando Stable Diffusion WebUI...")
    !git clone --depth 1 https://github.com/AUTOMATIC1111/stable-diffusion-webui

# 3. Pre-instalar Repositorios (Bypass Errores Git)
log("🔧 Configurando entorno de repositorios...")
repo_path = "stable-diffusion-webui/repositories"
!mkdir -p {repo_path}

sub_repos = {
    "stable-diffusion-stability-ai": "https://github.com/Stability-AI/stablediffusion.git",
    "stable-diffusion-webui-assets": "https://github.com/AUTOMATIC1111/stable-diffusion-webui-assets.git",
    "generative-models": "https://github.com/Stability-AI/generative-models.git",
    "k-diffusion": "https://github.com/crowsonkb/k-diffusion.git",
    "BLIP": "https://github.com/salesforce/BLIP.git"
}

for name, url in sub_repos.items():
    dest = os.path.join(repo_path, name)
    if not os.path.exists(dest):
        log(f"  -> Descargando {name}...")
        !git clone --depth 1 {url} {dest}

# 4. Descargar Modelo Principal
target_url = model_map[MODEL_TYPE]
log(f"⬇️ Descargando Modelo: {MODEL_TYPE} (6.5GB)...")
!aria2c -x 16 -s 16 -k 1M -d stable-diffusion-webui/models/Stable-diffusion -o model.safetensors "{target_url}"

# 5. Descargar LoRAs
log("⬇️ Descargando LoRAs seleccionadas...")
def dl_lora(name, data):
    fname = name.replace(" ", "_") + ".safetensors"
    !aria2c -x 16 -s 16 -k 1M -d stable-diffusion-webui/models/Lora -o "{fname}" "{data['url']}"

if LORA_SELECT == "Todas (Descargar las 3)":
    for n, d in lora_map.items(): dl_lora(n, d)
elif LORA_SELECT != "Ninguna":
    dl_lora(LORA_SELECT, lora_map[LORA_SELECT])

# 6. Cloudflared
log("☁️ Instalando Cloudflared...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

log("✅ Instalación terminada. Ejecuta la siguiente celda.")

In [ ]:
# @title 3. Ejecutar Sistema

def start_sd():
    log("🚀 Arrancando Stable Diffusion... (Espera los logs de carga abajo)")
    # Argumentos para máxima compatibilidad con Colab T4 y Python 3.12
    # --opt-sdp-attention: Reemplaza xformers porque xformers falla al compilar en Python 3.12
    # --no-half-vae: Necesario para evitar imagenes negras/puntos en modelos XL
    # --skip-python-version-check: Ignora la queja de A1111 sobre Python 3.12
    cmd = "cd stable-diffusion-webui && python launch.py --nowebui --opt-sdp-attention --api --listen --port 7860 --no-half-vae --enable-insecure-extension-access --skip-python-version-check"
    
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in iter(process.stdout.readline, ''):
        print(line, end='')

def update_npoint(url):
    if not NPOINT_ID: return
    target = f"https://api.npoint.io/{NPOINT_ID.split('/')[-1]}"
    try:
        data = requests.get(target).json()
        if not isinstance(data, dict): data = {}
        data["sd_url"] = url
        data["updated_at_sd"] = time.time()
        requests.post(target, json=data)
        log(f"✅ URL sincronizada en Npoint: {url}")
    except Exception as e: log(f"❌ Error Npoint: {e}")

def start_tunnel():
    log("⏳ Esperando a que el puerto de SD esté listo...")
    while True:
        try:
            requests.get("http://localhost:7860/docs")
            log("✅ Puerto detectado. Creando túnel...")
            break
        except: time.sleep(5)
    
    proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:7860"], stderr=subprocess.PIPE, text=True)
    regex = r"https://[a-zA-Z0-9-]+\.trycloudflare\.com"
    for line in iter(proc.stderr.readline, ''):
        match = re.search(regex, line)
        if match:
            url = match.group(0)
            log(f"🎨 IMAGE API URL: {url}")
            update_npoint(url)
            break

# Lanzar procesos
threading.Thread(target=start_sd, daemon=True).start()
start_tunnel()